# 🗜️ اضغطها | Media Lite

**ضغط ذكي للفيديو والصوت.**

1. ابعت ملف الفيديو أو الصوت في قناة **🗜️ اضغطها | Media Lite** على تيليجرام.
2. ارجع هنا واضغط **▶ تشغيل**.
3. النتيجة هترجع لنفس القناة تلقائيًا.
4. في آخر التشغيل هيظهر زر **📥 Import logs to Drive** لحفظ سجل التشغيل اختياريًا، وبعد الحفظ تقدر تفتحه أو تمسحه من الزر الظاهر.

**إعادة نتيجة قديمة:** اعمل Reply عليها واكتب `1` لنفس الإعداد، `2` لأصغر حجم، `3` للصوت فقط، أو رقم الحجم مثل `80`.

> أول تشغيل فقط هيطلب بيانات تيليجرام، وبعدها الإعدادات والجلسة تتحفظ في Google Drive الخاص بك.


In [ ]:
#@title ▶ تشغيل
#@markdown الوضع الافتراضي مناسب لمعظم الاستخدامات. غيّره فقط عند الحاجة.
#@markdown خانة **الحجم** اختيارية؛ استخدمها فقط مع «حجم فيديو محدد»، مثال: 80.
الوضع = "تلقائي — مناسب لمعظم الاستخدامات" #@param ["تلقائي — مناسب لمعظم الاستخدامات", "صوت صغير جدًا", "فيديو متوازن", "فيديو سريع", "أصغر حجم للفيديو", "حجم فيديو محدد"]
الحجم = "" #@param {type:"string"}

import os
import re
import subprocess
import urllib.request
import urllib.error
import json
import time

REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "telegram-smart-compressor/engine.py"
ENGINE_PATH = "/content/telegram_smart_compressor_engine.py"

PROFILE_MAP = {
    "تلقائي — مناسب لمعظم الاستخدامات": "SMART_AUTO",
    "صوت صغير جدًا": "AUDIO_TINY",
    "فيديو متوازن": "VIDEO_BALANCED",
    "فيديو سريع": "VIDEO_FAST",
    "أصغر حجم للفيديو": "VIDEO_SMALLEST",
    "حجم فيديو محدد": "VIDEO_TARGET_SIZE",
}

os.environ["TSC_PROFILE"] = PROFILE_MAP.get(الوضع, "SMART_AUTO")
os.environ["TSC_TARGET_SIZE_MB"] = str(الحجم or "").strip()
os.environ["TSC_UPDATE_CHANNEL"] = BRANCH

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha

    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "Media-Lite-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

def download_engine(sha):
    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Smart-Compressor-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return response.read().decode("utf-8")

try:
    print("🔄 جاري تحميل أحدث نسخة مستقرة...")
    sha = latest_sha()
    os.environ["TSC_ENGINE_SHA"] = sha
    engine = download_engine(sha)
    if len(engine) < 5000:
        raise RuntimeError("engine-too-small")
    compiled = compile(engine, ENGINE_PATH, "exec")
    print(f"✅ تم تحميل المحرك — commit {sha[:10]}")
    exec(compiled, globals(), globals())
except urllib.error.HTTPError as exc:
    print(f"❌ E001 — تعذر الوصول إلى GitHub (HTTP {exc.code}). جرّب مرة أخرى.")
except urllib.error.URLError:
    print("❌ E001 — تعذر الاتصال بـ GitHub. تأكد من الإنترنت وجرب مرة أخرى.")
except SyntaxError:
    print("❌ E002 — النسخة المنشورة فيها خطأ برمجي. لا تغيّر أي إعدادات وأرسل كود E002.")
except Exception as exc:
    print("❌ E003 — تعذر بدء المحرك.")
    print(f"التفاصيل المختصرة: {type(exc).__name__}: {str(exc)[:180]}")


## ☁️ 1 — من Google Drive إلى Telegram

**عايز تبعت فولدر أو ملف من Drive إلى تيليجرام؟** استخدم زر التشغيل تحت العنوان ده فقط.

الصق **رابط الملف أو الفولدر** في خانة **رابط Drive**. القناة الخاصة باسم **📁 Media Lite | File Transfer** هتتعمل تلقائيًا لو مش موجودة، أو يستخدم القناة الحالية.

سيب **عدد الملفات = 0** علشان ينقل كل الملفات، وفَعّل المجلدات الفرعية لو عايز محتويات الفولدر كاملة. جرب **معاينة فقط** الأول، وبعدها غيّر إلى **تنفيذ النقل**.

**تسجيل الدخول أول مرة:** اختر **QR بدون كود** لو رسالة تسجيل الدخول مش واصلة. هيظهر QR في Colab، افتح Telegram → الإعدادات → الأجهزة → ربط جهاز على جهاز مسجل دخول وامسحه. لو Colab وTelegram على نفس الجهاز، جرّب رابط الموافقة اللي هيظهر تحت الرمز. تقدر تختار **كود داخل Telegram** لو الكود بيوصلك في المحادثة الرسمية. بعد النجاح الجلسة بتتحفظ على Drive.


In [ ]:
#@title ☁️ تشغيل: Drive → Telegram
#@markdown **رابط Drive:** رابط ملف أو فولدر هيتنقل لقناة Telegram الخاصة.
رابط_Drive = "" #@param {type:"string"}
#@markdown **0 = جميع الملفات**. اختار رقم أكبر لو عايز دفعة صغيرة.
عدد_الملفات = 0 #@param {type:"integer"}
شامل_المجلدات_الفرعية = True #@param {type:"boolean"}
#@markdown **أول دخول Telegram:** لو الكود مش بيوصلك، استخدم QR ووافق من تطبيق Telegram.
طريقة_الدخول = "QR بدون كود" #@param ["QR بدون كود", "كود داخل Telegram"]
التشغيل = "معاينة فقط" #@param ["معاينة فقط", "تنفيذ النقل"]

import json
import re
import subprocess
import time
import urllib.request
import types

_TRANSFER_REPO = "abdullahsamirashour/gpt"
_TRANSFER_BRANCH = "main"
_TRANSFER_ENGINE = "telegram-smart-compressor/transfer_engine.py"

def _ml_transfer_latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_TRANSFER_REPO}.git",
         f"refs/heads/{_TRANSFER_BRANCH}"],
        capture_output=True, text=True, timeout=30,
    )
    sha = p.stdout.strip().split()[0] if p.returncode == 0 and p.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", sha):
        return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_TRANSFER_REPO}/git/ref/heads/{_TRANSFER_BRANCH}?v={int(time.time())}",
        headers={"User-Agent": "Media-Lite-Transfer-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

print("🔄 تحميل أحدث محرك نقل من GitHub...")
_transfer_sha = _ml_transfer_latest_sha()
_transfer_url = f"https://raw.githubusercontent.com/{_TRANSFER_REPO}/{_transfer_sha}/{_TRANSFER_ENGINE}"
with urllib.request.urlopen(
    urllib.request.Request(_transfer_url, headers={"User-Agent": "Media-Lite-Transfer-Colab"}),
    timeout=45,
) as _response:
    _transfer_code = _response.read().decode("utf-8")
if len(_transfer_code) < 10000:
    raise RuntimeError("محرك النقل من GitHub غير مكتمل.")
_transfer_module = types.ModuleType("media_lite_transfer")
exec(compile(_transfer_code, _TRANSFER_ENGINE, "exec"), _transfer_module.__dict__)
print(f"✅ المحرك جاهز — commit {_transfer_sha[:10]}")
_transfer_module.run_transfer(
    direction="drive_to_telegram",
    source=رابط_Drive.strip(),
    destination="",
    max_files=عدد_الملفات,
    recursive=شامل_المجلدات_الفرعية,
    preview=التشغيل == "معاينة فقط",
    login_method="qr" if طريقة_الدخول == "QR بدون كود" else "code",
)


## 📥 2 — من Telegram إلى Google Drive

**عايز تحفظ ملفات قناة تيليجرام على Drive؟** استخدم زر التشغيل تحت العنوان ده فقط.

**قناة Telegram المصدر:** لو فاضية هيستخدم قناة **📁 Media Lite | File Transfer**. لو عايز قناة تانية، اكتب `@username` أو معرف القناة المتاح لحسابك.

**فولدر Drive للحفظ:** لو فاضي هينشئ أو يستخدم مجلد **Media Lite Transfers**. لو عندك مجلد معين، الصق رابطه.

سيب **عدد الملفات = 0** للكل، وجرب المعاينة الأول. الملفات الأصلية مش بتتمسح أو تتضغط.

**تسجيل الدخول أول مرة:** اختر **QR بدون كود** لو رسالة تسجيل الدخول مش واصلة. هيظهر QR في Colab، افتح Telegram → الإعدادات → الأجهزة → ربط جهاز على جهاز مسجل دخول وامسحه. لو Colab وTelegram على نفس الجهاز، جرّب رابط الموافقة اللي هيظهر تحت الرمز. تقدر تختار **كود داخل Telegram** لو الكود بيوصلك في المحادثة الرسمية. بعد النجاح الجلسة بتتحفظ على Drive.


In [ ]:
#@title 📥 تشغيل: Telegram → Drive
#@markdown **قناة المصدر:** اتركها فاضية لقناتنا الخاصة أو اكتب @username لقناة أخرى.
قناة_Telegram_المصدر = "" #@param {type:"string"}
#@markdown **فولدر Drive:** اتركه فاضي لمجلد Media Lite Transfers أو الصق رابط فولدر.
فولدر_Drive_للحفظ = "" #@param {type:"string"}
#@markdown **0 = جميع الملفات الجديدة**.
عدد_الملفات = 0 #@param {type:"integer"}
#@markdown **أول دخول Telegram:** لو الكود مش بيوصلك، استخدم QR ووافق من تطبيق Telegram.
طريقة_الدخول = "QR بدون كود" #@param ["QR بدون كود", "كود داخل Telegram"]
التشغيل = "معاينة فقط" #@param ["معاينة فقط", "تنفيذ النقل"]

import json
import re
import subprocess
import time
import urllib.request
import types

_TRANSFER_REPO = "abdullahsamirashour/gpt"
_TRANSFER_BRANCH = "main"
_TRANSFER_ENGINE = "telegram-smart-compressor/transfer_engine.py"

def _ml_transfer_latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_TRANSFER_REPO}.git",
         f"refs/heads/{_TRANSFER_BRANCH}"],
        capture_output=True, text=True, timeout=30,
    )
    sha = p.stdout.strip().split()[0] if p.returncode == 0 and p.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", sha):
        return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_TRANSFER_REPO}/git/ref/heads/{_TRANSFER_BRANCH}?v={int(time.time())}",
        headers={"User-Agent": "Media-Lite-Transfer-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

print("🔄 تحميل أحدث محرك نقل من GitHub...")
_transfer_sha = _ml_transfer_latest_sha()
_transfer_url = f"https://raw.githubusercontent.com/{_TRANSFER_REPO}/{_transfer_sha}/{_TRANSFER_ENGINE}"
with urllib.request.urlopen(
    urllib.request.Request(_transfer_url, headers={"User-Agent": "Media-Lite-Transfer-Colab"}),
    timeout=45,
) as _response:
    _transfer_code = _response.read().decode("utf-8")
if len(_transfer_code) < 10000:
    raise RuntimeError("محرك النقل من GitHub غير مكتمل.")
_transfer_module = types.ModuleType("media_lite_transfer")
exec(compile(_transfer_code, _TRANSFER_ENGINE, "exec"), _transfer_module.__dict__)
print(f"✅ المحرك جاهز — commit {_transfer_sha[:10]}")
_transfer_module.run_transfer(
    direction="telegram_to_drive",
    source=قناة_Telegram_المصدر.strip(),
    destination=فولدر_Drive_للحفظ.strip(),
    max_files=عدد_الملفات,
    recursive=True,
    preview=التشغيل == "معاينة فقط",
    login_method="qr" if طريقة_الدخول == "QR بدون كود" else "code",
)
